# 02 · ECFP Baselines — RandomForest + ECFP-MLP
RDKit ECFP(Morgan r=2, 2048bit) 위에서 **RandomForest 베이스라인**과 **ECFP-MLP(멀티태스크)**를 회귀·분류 모두에 대해 학습/평가.
회귀는 z-score 정규화 후 학습, 평가 시 원단위 역변환.

In [ ]:
# --- Colab 자가치유 셋업 (로컬/CI에서는 자동 스킵) ---
import os, sys, subprocess
try:
    import src  # repo 루트에서 실행 중이면 바로 import
except ModuleNotFoundError:
    if not os.path.exists("admet-property-predictor"):
        subprocess.run(["git", "clone",
            "https://github.com/zqvo04/admet-property-predictor.git"], check=False)
    os.chdir("admet-property-predictor")
    subprocess.run(["bash", "setup_colab.sh"], check=False)
    import src
print("src", src.__version__, "| cwd", os.getcwd())

In [ ]:
# === CONFIG (Colab에서 값만 키우면 전체 재현) ===
# 검증용 기본값은 작게 설정 — 실제 학습 시 EPOCHS↑, 엔드포인트 추가.
import os
TDC_PATH = "data/"
EPOCHS   = int(os.environ.get("ADMET_EPOCHS", 2))   # Colab 실전: 100
REG_EP   = ["Caco2"]          # 회귀 엔드포인트 (확장: Solubility, Lipophilicity, ...)
CLS_EP   = ["hERG"]           # 분류 엔드포인트 (확장: BBBP, CYP3A4_Inhibition, ...)
ENDPOINTS_USED = REG_EP + CLS_EP
SEED = 1
print("endpoints:", ENDPOINTS_USED, "| epochs:", EPOCHS)

In [ ]:
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from src.data import load_tdc_endpoint, build_multitask, ENDPOINTS
from src.featurizers import ecfp_featurize
from src.models import build_model
from src.train import Trainer, TrainConfig, ECFPDataset
from src.evaluate import evaluate_multitask, summarize, regression_metrics, classification_metrics
np.random.seed(SEED); torch.manual_seed(SEED)
data = {ep: load_tdc_endpoint(ep, seed=SEED, tdc_path=TDC_PATH) for ep in ENDPOINTS_USED}
data

## (A) RandomForest 베이스라인 (per-endpoint)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
rf_rows = []
for ep, d in data.items():
    Xtr = ecfp_featurize(d.smiles('train')); Xte = ecfp_featurize(d.smiles('test'))
    if d.task_type == 'reg':
        rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=SEED)
        rf.fit(Xtr, d.targets('train')); pred = rf.predict(Xte)
        m = regression_metrics(d.targets('test'), pred)
    else:
        rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
        rf.fit(Xtr, d.targets('train')); pred = rf.predict_proba(Xte)[:,1]
        m = classification_metrics(d.targets('test'), pred)
    m.update({'endpoint': ep, 'model': 'RandomForest', 'metric': ENDPOINTS[ep]['metric'],
              'official': m.get(ENDPOINTS[ep]['metric'])})
    rf_rows.append(m)
rf_df = pd.DataFrame(rf_rows); rf_df

## (B) ECFP-MLP 멀티태스크 (회귀+분류 통합)

In [ ]:
task_names = list(data.keys())
task_types = [data[n].task_type for n in task_names]
type_vec = [0 if t=='reg' else 1 for t in task_types]
mt_tr = build_multitask(data, 'train'); mt_va = build_multitask(data, 'valid'); mt_te = build_multitask(data, 'test')
def to_loader(mt, shuffle):
    X = ecfp_featurize(mt.smiles)
    return DataLoader(ECFPDataset(X, mt.Y, mt.mask), batch_size=64, shuffle=shuffle), mt
tl,_ = to_loader(mt_tr, True); vl,_ = to_loader(mt_va, False); tel, mt_te = to_loader(mt_te, False)
model = build_model('ecfp', num_tasks=len(task_names))
trainer = Trainer(model, task_types=type_vec, cfg=TrainConfig(epochs=EPOCHS,
                  ckpt_dir='checkpoints/', run_name='ecfp_mlp'))
trainer.fit(tl, vl)
pred = trainer.predict(tel)
res = evaluate_multitask(pred, mt_te.Y, mt_te.mask, task_names, task_types, mt_te.scalers)
summarize(res, ENDPOINTS)

In [ ]:
trainer.save_curve()  # 학습곡선 저장

✅ ECFP RandomForest + MLP 베이스라인 완료. 다음: `03_GraphConv_GIN`.